In [ ]:
import os
import warnings
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv

from openai import OpenAI
import google.generativeai as genai
from anthropic import Anthropic

#load environment variables
load_dotenv()
print("Attempting to load API keys from .env file...")

#load keys
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

#Configure clients
openai_client = OpenAI(api_key= openai_api_key)

genai.configure(api_key= gemini_api_key)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")


claude_client = Anthropic(api_key= anthropic_api_key)


In [ ]:
def print_markdown(text):
    display(Markdown(text))

In [ ]:
def display_html_code(provider_name, html_content):
    print_markdown(f"### Generated HTML from {provider_name}:")
    display(Markdown(f"'''html\n{html_content}\n'''"))

In [ ]:
#let's test the Math capabilities of these 3 LLMs
test_prompt= "A father is 36 years old, and his son is 6 years old. In how many years will the father be exactly five times as old as his son?"

In [ ]:
#let's test their creativity!
test_prompt = "Viết 1 bài thơ thú vị cho bạn của tôi khi mới 1 tuổi."

In [ ]:
response = openai_client.chat.completions.create(model = "gpt-4o",
                                                messages = [{"role": "user",
                                                            "content": test_prompt}],
                                                temperature = 0.5)
print_markdown(response.choices[0].message.content)

In [ ]:
response = gemini_model.generate_content(test_prompt)
print_markdown(response.text)

In [ ]:
from anthropic import BadRequestError

try:
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": test_prompt}]
    )
    print_markdown(response.content[0].text)

except BadRequestError as e:
    print("Claude API error:", e)
    print("Check Anthropic billing/credits, or use Gemini for now.")

In [ ]:
# Define the startup name and concept
startup_name = "ConnectGenius"
startup_concept = "An intelligent CRM system that uses AI to analyze customer interactions, predict needs, and automate personalized follow-ups. Focus on improving customer retention and sales efficiency for businesses of all sizes."

In [ ]:
# Define the core prompt for the LLMs
# We explicitly ask for HTML code and specify the file name 'index.html'
html_prompt = f"""
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new startup.

Startup Name: {startup_name}
Concept: {startup_concept}

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

-- Don't include images in the code. Raw html code with inline css for styling.

1. A sleek header with the startup name in a bold, modern font and a compelling tagline
2. A hero section with a clear value proposition and call-to-action button
3. A features section highlighting 3-4 key benefits with icons or simple visuals
4. A "How it Works" section with numbered steps
5. A testimonials section with fictional customer quotes
6. A pricing section with at least two tiers
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern color palette (primary, secondary, and accent colors). 
Include responsive design elements, subtle animations, and whitespace for readability.
Emphasize AI capabilities, ease of use, and business benefits throughout the copy.
Focus on conversion-optimized marketing messages that highlight pain points and solutions.

Do not include any explanations before or after the code block. Just provide the raw HTML code.
"""

print_markdown("**Core Prompt defined for the LLMs:**")
print_markdown(f"> {html_prompt}")  # Print the start of the prompt to verify

In [ ]:
# Let's generate HTML using OpenAI API, we will use gpt-4o model

openai_html_output = "<!-- OpenAI generation not run or failed -->"

print_markdown("## Calling OpenAI API...")
try:
    response = openai_client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role": "user", "content": html_prompt}
        ],
        temperature = 0.5,
    )
    openai_html_output = response.choices[0].message.content

    if openai_html_output.strip().startswith("```html"):
        lines = openai_html_output.strip().splitlines()
        openai_html_output = "\n".join(lines[1:-1]).strip()
    else:
        openai_html_output = openai_html_output.strip()

    # Display the generated HTML code
    display_html_code("OpenAI (gpt-4o)", openai_html_output)

    # Let's Save the output to a file
    file_path = "openai_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(openai_html_output)
    print_markdown(f"Successfully saved OpenAI output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling OpenAI API: {e}")
    openai_html_output = f"<!-- Error calling OpenAI API: {e} -->"

In [ ]:
gemini_html_output = "<!-- Gemini generation not run or failed -->"

print_markdown("## Calling Google Gemini API...")
try:
    response = gemini_model.generate_content(
        html_prompt,
    )

    raw_output = response.text
    if raw_output.strip().startswith("```html"):
        # Remove the first line (```html) and the last line (```)
        lines = raw_output.strip().splitlines()
        gemini_html_output = "\n".join(lines[1:-1]).strip()
    else:
        gemini_html_output = raw_output.strip()

    # Display the generated HTML code
    display_html_code("Google Gemini (gemini-2.5-flash)", gemini_html_output)

    # --- Save the output to a file ---
    file_path = "gemini_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(gemini_html_output)
    print_markdown(f"Successfully saved Gemini output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling Google Gemini API: {e}")
    gemini_html_output = f"<!-- Error calling Google Gemini API: {e} -->"


In [ ]:
# Generate HTML using Anthropic Claude

claude_html_output = "<!-- Claude generation not run or failed -->"

print_markdown("## Calling Anthropic Claude API...")

claude_model_name = "claude-3-7-sonnet-20250219"
print_markdown(f"(Using model: {claude_model_name})")

try:
    response = claude_client.messages.create(
        model = claude_model_name,
        max_tokens = 20000,
        messages = [{"role": "user", "content": html_prompt}],
    )
    raw_output = response.content[0].text
    if raw_output.strip().startswith("```html"):
        lines = raw_output.strip().splitlines()
        claude_html_output = "\n".join(lines[1:-1]).strip()
    else:
        claude_html_output = raw_output.strip()

    # Display the generated HTML code
    display_html_code(f"Anthropic Claude ({claude_model_name})", claude_html_output)

    # --- Save the output to a file ---
    file_path = "claude_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(claude_html_output)
    print_markdown(f"Successfully saved Claude output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling Anthropic Claude API: {e}")
    claude_html_output = f"<!-- Error calling Anthropic Claude API: {e} -->"